In [ ]:
!pip install -q nibabel pydicom kagglehub scikit-image seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import cv2
import warnings
import shutil
from scipy import ndimage
from scipy.stats import entropy, kstest, mannwhitneyu, chi2_contingency, iqr
from skimage.feature import graycomatrix, graycoprops
from collections import defaultdict
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.stats import pearsonr
import nibabel as nib
from PIL import Image
import kagglehub

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 16
torch.manual_seed(SEED)
np.random.seed(SEED)

print("\nDownloading MosMedData")
mosmed_path = kagglehub.dataset_download("mathurinache/mosmeddata-chest-ct-scans-with-covid19")
print(f"MosMedData path: {mosmed_path}")

# Physics Constraints and HU-Attenuation Conversion

In [ ]:
class PhysicsConstants:
    MU_WATER_70KEV = 0.190  
    HU_MIN = -1000
    HU_MAX = 400

    @staticmethod
    def hu_to_mu(hu_values):
        return PhysicsConstants.MU_WATER_70KEV * (1 + hu_values / 1000.0)


    @staticmethod
    def validate_physics(hu_slice, tissue_type='lung'):
        ranges = {
            'lung': (-900, -500),
            'ggo': (-500, -100),
            'consolidation': (0, 100)
        }
        hu_min, hu_max = ranges[tissue_type]
        mask = (hu_slice >= hu_min) & (hu_slice <= hu_max)
        
        if mask.sum() > 0:
            hu_tissue = hu_slice[mask]
            mu_tissue = PhysicsConstants.hu_to_mu(hu_tissue)
            print(f"  {tissue_type.upper():15s}: HU [{hu_tissue.min():.0f}, {hu_tissue.max():.0f}] "
                  f"→ μ [{mu_tissue.min():.4f}, {mu_tissue.max():.4f}] cm⁻¹")

def simple_lung_mask(hu_slice):
    mask = ((hu_slice >= -900) & (hu_slice <= 100)).astype(np.float32)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask

# Lung Mask Visualization

In [ ]:
def visualize_lung_mask(hu_slice, title="Lung Segmentation"):
    mask = simple_lung_mask(hu_slice)
    masked_image = hu_slice * mask
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original Image 
    im1 = axes[0].imshow(hu_slice, cmap='gray', vmin=-1000, vmax=400)
    axes[0].set_title('Original CT (HU)')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    
    # Lung mask
    im2 = axes[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('Lung Mask')
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    
    # Masked image
    im3 = axes[2].imshow(masked_image, cmap='gray', vmin=-1000, vmax=400)
    axes[2].set_title('Masked Lung Region')
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\n{title} Statistics:")
    print(f"  Mask coverage: {mask.sum() / mask.size * 100:.2f}%")
    if mask.sum() > 0:
        print(f"  Lung HU range: [{masked_image[mask > 0].min():.0f}, {masked_image[mask > 0].max():.0f}]")
        
        print("\nPhysics Validation:")
        PhysicsConstants.validate_physics(masked_image, 'lung')
        PhysicsConstants.validate_physics(masked_image, 'ggo')
        PhysicsConstants.validate_physics(masked_image, 'consolidation')

# Physics Feature Extraction

In [ ]:
def extract_physics_features(slice_array, mask_array, compute_texture=False):
    '''
    Paramters:

    slice_Array : np.ndarray (H,W)
        CT slice in HU values ( Not normalized to attentuation value , but original HU values)
    mask_Array : A binary mask (0,1) for the lung tissue 
    Compute_texture : Wether to compute GCLM texture features 

    return:
    dict : Dictionary of features 
    '''
    mask = (mask_array > 0.5).astype(bool)
    masked_hu = slice_array[mask]

    if masked_hu.size == 0:
        return {
            'mu_avg': np.nan,
            'hu_std': np.nan,
            'hu_p10': np.nan,
            'hu_p25': np.nan,
            'hu_p50': np.nan,
            'hu_p75': np.nan,
            'hu_p90': np.nan,
            'grad_mean': np.nan,
            'grad_std': np.nan,
            'mask_area_pixels': 0,
            'mask_fraction': 0.0,
            'entropy': np.nan,
            'contrast': np.nan,
            'homogeneity': np.nan
        }

    features = {
        'mu_avg': float(np.mean(masked_hu)),
        'hu_std': float(np.std(masked_hu)),
        'hu_p10': float(np.percentile(masked_hu, 10)),
        'hu_p25': float(np.percentile(masked_hu, 25)),
        'hu_p50': float(np.percentile(masked_hu, 50)),
        'hu_p75': float(np.percentile(masked_hu, 75)),
        'hu_p90': float(np.percentile(masked_hu, 90)),
        'mask_area_pixels': int(mask.sum()),
        'mask_fraction': float(mask.sum() / mask.size)
    }

    grad_y = ndimage.sobel(slice_array,axis=0)
    grad_x = ndimage.sobel(slice_array,axis = 1)
    grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)

    masked_grad = grad_magnitude[mask]
    features['grad_mean'] = float(np.mean(masked_grad))
    features['grad_std'] = float(np.std(masked_grad))

    if compute_texture:
        try:
            
            hu_norm = np.clip(slice_array, -1000, 400)
            hu_norm = ((hu_norm + 1000) / 1400 * 255).astype(np.uint8)
            
            
            hu_masked = np.where(mask, hu_norm, 0)
            
            
            glcm = graycomatrix(hu_masked, distances=[1], 
                               angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                               levels=256, symmetric=True, normed=True)
            
            
            features['contrast'] = float(np.mean(graycoprops(glcm, 'contrast')))
            features['homogeneity'] = float(np.mean(graycoprops(glcm, 'homogeneity')))
            
            
            hist, _ = np.histogram(masked_hu, bins=50, density=True)
            hist = hist[hist > 0]  # Remove zeros
            features['entropy'] = float(entropy(hist))
            
        except Exception as e:
            features['contrast'] = np.nan
            features['homogeneity'] = np.nan
            features['entropy'] = np.nan
    else:
        features['contrast'] = np.nan
        features['homogeneity'] = np.nan
        features['entropy'] = np.nan
    
    return features

    
    
                                
                                     
    

# Data Processing 


In [ ]:

def process_datasets_merged(mosmed_path, max_per_volume=30, extract_features=True, max_samples_per_class=2500):
    
    mosmed_normal_files = list(Path(mosmed_path).glob("**/CT-0/**/*.nii*"))
    print(f"Found {len(mosmed_normal_files)} NORMAL NIfTI volumes")
    print(f"Target: {max_samples_per_class:,} NORMAL samples\n")
    
    normal_count = 0
    normal_volumes_processed = 0
    
    for vol_idx, nifti_file in enumerate(tqdm(mosmed_normal_files, desc="Processing NORMAL")):
        if normal_count >= max_samples_per_class:
            break
        
        try:
            # Load NIfTI file
            nii = nib.load(str(nifti_file))
            volume = nii.get_fdata()
            
            # Validate volume
            if volume.size == 0:
                log_error(f"Empty volume: {nifti_file.name}")
                continue
            
            if len(volume.shape) != 3:
                log_error(f"Invalid shape {volume.shape}: {nifti_file.name}")
                continue
            
            n_slices = volume.shape[2]
            
            if n_slices < 10:
                log_error(f"Too few slices ({n_slices}): {nifti_file.name}")
                continue
            
            # Select middle slices (most informative for chest CT)
            start_idx = n_slices // 4
            end_idx = 3 * n_slices // 4
            num_slices_to_extract = min(max_per_volume, end_idx - start_idx)
            
            slice_indices = np.linspace(start_idx, end_idx - 1, 
                                       num_slices_to_extract, 
                                       dtype=int)
            
            slices_added = 0
            
            for slice_idx in slice_indices:
                if normal_count >= max_samples_per_class:
                    break
                
                # Extract slice
                hu_slice = volume[:, :, slice_idx].copy()
                
                # Clip to valid HU range
                hu_slice = np.clip(hu_slice, PhysicsConstants.HU_MIN, 
                                  PhysicsConstants.HU_MAX)
                
                # Generate lung mask
                mask = simple_lung_mask(hu_slice)
                
                # Skip if mask is too small (likely not a useful slice)
                if mask.sum() < 1000:
                    continue
                
                # Convert HU to attenuation coefficient (μ)
                mu_slice = PhysicsConstants.hu_to_mu(hu_slice)
                
                # Normalize CT for storage and neural network input
                # Map [-1000, 400] HU to [-1, 1]
                ct_normalized = (hu_slice - PhysicsConstants.HU_MIN) / \
                               (PhysicsConstants.HU_MAX - PhysicsConstants.HU_MIN) * 2 - 1
                
                # Extract physics-based features
                phys_features = extract_physics_features(
                    hu_slice, mask, compute_texture=extract_features
                )
                
                # Create unique file ID
                file_id = f"mosmed_normal_{vol_idx:04d}_slice_{slice_idx:03d}"
                
                # Save arrays
                np.save(f"{OUTPUT_DIR}/{file_id}_ct.npy", ct_normalized.astype(np.float32))
                np.save(f"{OUTPUT_DIR}/{file_id}_mu.npy", mu_slice.astype(np.float32))
                np.save(f"{OUTPUT_DIR}/{file_id}_mask.npy", mask.astype(np.float32))
                
                # Calculate statistics for masked region
                lung_pixels = mask > 0.5
                if lung_pixels.sum() > 0:
                    hu_avg = float(hu_slice[lung_pixels].mean())
                    mu_avg = float(mu_slice[lung_pixels].mean())
                else:
                    hu_avg = 0.0
                    mu_avg = 0.0
                
                # Create record
                record = {
                    'id': file_id,
                    'ct_path': f"{OUTPUT_DIR}/{file_id}_ct.npy",
                    'mu_path': f"{OUTPUT_DIR}/{file_id}_mu.npy",
                    'mask_path': f"{OUTPUT_DIR}/{file_id}_mask.npy",
                    'label': 0,  # Normal
                    'source': 'mosmed',
                    'hu_mean': hu_avg,
                    'mu_mean': mu_avg,
                    'volume_id': f"normal_{vol_idx:04d}",
                    'slice_id': int(slice_idx),
                    'original_file': nifti_file.name
                }
                
                # Add physics features
                record.update(phys_features)
                
                data_records.append(record)
                slices_added += 1
                normal_count += 1
            
            if slices_added > 0:
                normal_volumes_processed += 1
            
            # Clear memory
            del volume, nii
            
        except Exception as e:
            log_error(f"Error processing {nifti_file.name}: {str(e)[:200]}")
            continue
    
    print(f"\n✓ Processed {normal_count:,} NORMAL slices from {normal_volumes_processed} volumes")
    
    
    covid_patterns = [
        "**/CT-2/**/*.nii*",  # Moderate
    ]
    
    mosmed_covid_files = []
    for pattern in covid_patterns:
        mosmed_covid_files.extend(list(Path(mosmed_path).glob(pattern)))
    
    print(f"Found {len(mosmed_covid_files)} COVID NIfTI volumes")
    print(f"Target: {max_samples_per_class:,} COVID samples\n")
    
    covid_count = 0
    covid_volumes_processed = 0
    
    for vol_idx, nifti_file in enumerate(tqdm(mosmed_covid_files, desc="Processing COVID")):
        if covid_count >= max_samples_per_class:
            break
        
        try:
            # Load NIfTI file
            nii = nib.load(str(nifti_file))
            volume = nii.get_fdata()
            
            # Validate volume
            if volume.size == 0:
                log_error(f"Empty volume: {nifti_file.name}")
                continue
            
            if len(volume.shape) != 3:
                log_error(f"Invalid shape {volume.shape}: {nifti_file.name}")
                continue
            
            n_slices = volume.shape[2]
            
            if n_slices < 10:
                log_error(f"Too few slices ({n_slices}): {nifti_file.name}")
                continue
            
            # Select middle slices
            start_idx = n_slices // 4
            end_idx = 3 * n_slices // 4
            num_slices_to_extract = min(max_per_volume, end_idx - start_idx)
            
            slice_indices = np.linspace(start_idx, end_idx - 1, 
                                       num_slices_to_extract, 
                                       dtype=int)
            
            slices_added = 0
            
            for slice_idx in slice_indices:
                if covid_count >= max_samples_per_class:
                    break
                
                # Extract slice
                hu_slice = volume[:, :, slice_idx].copy()
                
                # Clip to valid HU range
                hu_slice = np.clip(hu_slice, PhysicsConstants.HU_MIN, 
                                  PhysicsConstants.HU_MAX)
                
                # Generate lung mask
                mask = simple_lung_mask(hu_slice)
                
                # Skip if mask is too small
                if mask.sum() < 1000:
                    continue
                
                # Convert HU to attenuation coefficient (μ)
                mu_slice = PhysicsConstants.hu_to_mu(hu_slice)
                
                # Normalize CT
                ct_normalized = (hu_slice - PhysicsConstants.HU_MIN) / \
                               (PhysicsConstants.HU_MAX - PhysicsConstants.HU_MIN) * 2 - 1
                
                # Extract physics-based features
                phys_features = extract_physics_features(
                    hu_slice, mask, compute_texture=extract_features
                )
                
                # Create unique file ID
                file_id = f"mosmed_covid_{vol_idx:04d}_slice_{slice_idx:03d}"
                
                # Save arrays
                np.save(f"{OUTPUT_DIR}/{file_id}_ct.npy", ct_normalized.astype(np.float32))
                np.save(f"{OUTPUT_DIR}/{file_id}_mu.npy", mu_slice.astype(np.float32))
                np.save(f"{OUTPUT_DIR}/{file_id}_mask.npy", mask.astype(np.float32))
                
                # Calculate statistics
                lung_pixels = mask > 0.5
                if lung_pixels.sum() > 0:
                    hu_avg = float(hu_slice[lung_pixels].mean())
                    mu_avg = float(mu_slice[lung_pixels].mean())
                else:
                    hu_avg = 0.0
                    mu_avg = 0.0
                
                # Create record
                record = {
                    'id': file_id,
                    'ct_path': f"{OUTPUT_DIR}/{file_id}_ct.npy",
                    'mu_path': f"{OUTPUT_DIR}/{file_id}_mu.npy",
                    'mask_path': f"{OUTPUT_DIR}/{file_id}_mask.npy",
                    'label': 1,  # COVID
                    'source': 'mosmed',
                    'hu_mean': hu_avg,
                    'mu_mean': mu_avg,
                    'volume_id': f"covid_{vol_idx:04d}",
                    'slice_id': int(slice_idx),
                    'original_file': nifti_file.name
                }
                
                # Add physics features
                record.update(phys_features)
                
                data_records.append(record)
                slices_added += 1
                covid_count += 1
            
            if slices_added > 0:
                covid_volumes_processed += 1
            
            # Clear memory
            del volume, nii
            
        except Exception as e:
            log_error(f"Error processing {nifti_file.name}: {str(e)[:200]}")
            continue
    
    print(f"\n✓ Processed {covid_count:,} COVID slices from {covid_volumes_processed} volumes")
    
    # ========================================================================
    # VALIDATE AND CREATE DATAFRAME
    # ========================================================================
    if len(data_records) == 0:
        raise ValueError("No data processed! Check dataset paths and error log.")
    
    df = pd.DataFrame(data_records)
    
    print(f"\n{'='*70}")
    print(f"DATASET SUMMARY")
    print(f"{'='*70}")
    print(f"Total samples: {len(df):,}")
    print(f"  COVID:  {len(df[df['label']==1]):,} ({len(df[df['label']==1])/len(df)*100:.1f}%)")
    print(f"  Normal: {len(df[df['label']==0]):,} ({len(df[df['label']==0])/len(df)*100:.1f}%)")
    print(f"\nTotal volumes processed:")
    print(f"  COVID:  {covid_volumes_processed}")
    print(f"  Normal: {normal_volumes_processed}")
    
    # ========================================================================
    # SPLIT BY VOLUME (LEAK-FREE)
    # ========================================================================
    print(f"\n{'='*70}")
    print(f"SPLITTING DATASET BY VOLUME (70/15/15) - LEAK-FREE")
    print(f"{'='*70}")
    
    # Get unique volumes with their labels
    volume_info = df[['volume_id', 'label']].drop_duplicates().reset_index(drop=True)
    volume_ids = volume_info['volume_id'].values
    volume_labels = volume_info['label'].values
    
    print(f"\nTotal unique volumes: {len(volume_ids)}")
    print(f"  COVID volumes:  {(volume_labels == 1).sum()}")
    print(f"  Normal volumes: {(volume_labels == 0).sum()}")
    
    # First split: 85% train+val, 15% test
    train_val_ids, test_ids, train_val_labels, test_labels = train_test_split(
        volume_ids, volume_labels, 
        test_size=0.15, 
        random_state=SEED, 
        stratify=volume_labels
    )
    
    # Second split: split train+val into 70% train, 15% val (relative to original)
    # 0.15/0.85 ≈ 0.176 of train+val becomes validation
    train_ids, val_ids, _, _ = train_test_split(
        train_val_ids, train_val_labels,
        test_size=0.176,  # This gives us 15% of original data
        random_state=SEED,
        stratify=train_val_labels
    )
    
    # Create split dataframes
    train_df = df[df['volume_id'].isin(train_ids)].reset_index(drop=True)
    val_df = df[df['volume_id'].isin(val_ids)].reset_index(drop=True)
    test_df = df[df['volume_id'].isin(test_ids)].reset_index(drop=True)
    
    # Verify no volume leakage
    assert len(set(train_ids) & set(val_ids)) == 0, "Volume leakage: train-val"
    assert len(set(train_ids) & set(test_ids)) == 0, "Volume leakage: train-test"
    assert len(set(val_ids) & set(test_ids)) == 0, "Volume leakage: val-test"
    
    print(f"\n--- Volume Split ---")
    print(f"  Train volumes: {len(train_ids):,}")
    print(f"  Val volumes:   {len(val_ids):,}")
    print(f"  Test volumes:  {len(test_ids):,}")
    
    print(f"\n--- Resulting Slices ---")
    print(f"  Train: {len(train_df):,} slices")
    print(f"    COVID:  {len(train_df[train_df['label']==1]):,} ({len(train_df[train_df['label']==1])/len(train_df)*100:.1f}%)")
    print(f"    Normal: {len(train_df[train_df['label']==0]):,} ({len(train_df[train_df['label']==0])/len(train_df)*100:.1f}%)")
    
    print(f"  Val:   {len(val_df):,} slices")
    print(f"    COVID:  {len(val_df[val_df['label']==1]):,} ({len(val_df[val_df['label']==1])/len(val_df)*100:.1f}%)")
    print(f"    Normal: {len(val_df[val_df['label']==0]):,} ({len(val_df[val_df['label']==0])/len(val_df)*100:.1f}%)")
    
    print(f"  Test:  {len(test_df):,} slices")
    print(f"    COVID:  {len(test_df[test_df['label']==1]):,} ({len(test_df[test_df['label']==1])/len(test_df)*100:.1f}%)")
    print(f"    Normal: {len(test_df[test_df['label']==0]):,} ({len(test_df[test_df['label']==0])/len(test_df)*100:.1f}%)")
    
    # ========================================================================
    # SAVE CSVS
    # ========================================================================
    train_df.to_csv('/kaggle/working/train.csv', index=False)
    val_df.to_csv('/kaggle/working/val.csv', index=False)
    test_df.to_csv('/kaggle/working/test.csv', index=False)
    df.to_csv('/kaggle/working/full_balanced.csv', index=False)
    
    print(f"\n{'='*70}")
    print(f"✓ CSVs saved to /kaggle/working/")
    print(f"  - train.csv ({len(train_df):,} samples)")
    print(f"  - val.csv ({len(val_df):,} samples)")
    print(f"  - test.csv ({len(test_df):,} samples)")
    print(f"  - full_balanced.csv ({len(df):,} samples)")
    print(f"{'='*70}")
    
    if os.path.exists(error_log_path):
        print(f"Some errors occurred. Check: {error_log_path}")
    
    return train_df, val_df, test_df, df

In [ ]:
train_df, val_df, test_df, full_balanced_df = process_datasets_merged(
    mosmed_path,
    max_per_volume=30,
    extract_features=True,
    max_samples_per_class=2750
)

# Sanity Checks


In [ ]:
def run_all_sanity_checks(train_df, val_df, test_df, output_dir='/kaggle/working/sanity_checks'):
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f"RUNNING SANITY CHECKS")
    print(f"{'='*70}")
    
    full_df = pd.concat([train_df, val_df, test_df])
    
    
    print("\nCHECK 1: File Integrity")
    
    missing_ct = []
    missing_mu = []
    missing_mask = []
    
    for idx, row in full_df.iterrows():
        if not os.path.exists(row['ct_path']):
            missing_ct.append(row['id'])
        if not os.path.exists(row['mu_path']):
            missing_mu.append(row['id'])
        if not os.path.exists(row['mask_path']):
            missing_mask.append(row['id'])
    
    print(f"  Total samples: {len(full_df):,}")
    print(f"  Missing CT files: {len(missing_ct)}")
    print(f"  Missing μ files: {len(missing_mu)}")
    print(f"  Missing mask files: {len(missing_mask)}")
    
    # ===== CHECK 2: HU Range & Distribution =====
    print("\nCHECK 2: HU Range & Distribution")
    
    # Sample for efficiency
    sample_df = full_df.sample(n=min(1000, len(full_df)), random_state=42)
    
    hu_stats = []
    for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Scanning HU"):
        ct = np.load(row['ct_path'])
        # Denormalize to get original HU
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        hu_stats.append({
            'min': ct_hu.min(),
            'max': ct_hu.max(),
            'mean': ct_hu.mean(),
            'std': ct_hu.std()
        })
    
    stats_df = pd.DataFrame(hu_stats)
    
    print(f"  Global HU Statistics (n={len(sample_df)}):")
    print(f"    Min:  [{stats_df['min'].min():.1f}, {stats_df['min'].max():.1f}]")
    print(f"    Max:  [{stats_df['max'].min():.1f}, {stats_df['max'].max():.1f}]")
    print(f"    Mean: {stats_df['mean'].mean():.1f} ± {stats_df['mean'].std():.1f}")
    
    # Check for outliers and SAVE THE COUNT
    outliers_df = stats_df[(stats_df['min'] < -1500) | (stats_df['max'] > 3000)]
    outlier_count = len(outliers_df)
    if outlier_count > 0:
        print(f" WARNING: {outlier_count} slices with extreme HU values!")
    
    # Plot HU distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(stats_df['mean'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Mean HU per Slice')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Mean HU Values')
    axes[0].grid(alpha=0.3)
    
    axes[1].hist(stats_df['std'], bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[1].set_xlabel('Std HU per Slice')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of HU Standard Deviation')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check2_hu_distribution.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 3: Mask Integrity =====
    print("\nCHECK 3: Mask Integrity")
    
    mask_areas = full_df['mask_area_pixels'].values
    
    print(f"  Mask Area Statistics:")
    print(f"    Mean: {mask_areas.mean():.0f} pixels")
    print(f"    Median: {np.median(mask_areas):.0f} pixels")
    print(f"    Min: {mask_areas.min():.0f} pixels")
    print(f"    Max: {mask_areas.max():.0f} pixels")
    
    # Check for empty or tiny masks
    empty_masks = (mask_areas < 1000).sum()
    print(f"    Slices with <1000 pixels: {empty_masks} ({empty_masks/len(full_df)*100:.2f}%)")
    
    if empty_masks > len(full_df) * 0.02:
        print(f" WARNING: {empty_masks/len(full_df)*100:.1f}% of masks are too small!")
    
    # Plot mask area distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].boxplot([full_df[full_df['label']==0]['mask_area_pixels'].values,
                      full_df[full_df['label']==1]['mask_area_pixels'].values],
                     labels=['Normal', 'COVID'])
    axes[0].set_ylabel('Mask Area (pixels)')
    axes[0].set_title('Mask Area by Label')
    axes[0].grid(alpha=0.3)
    
    axes[1].hist([full_df[full_df['label']==0]['mask_area_pixels'].values,
                  full_df[full_df['label']==1]['mask_area_pixels'].values],
                 bins=50, label=['Normal', 'COVID'], alpha=0.6)
    axes[1].set_xlabel('Mask Area (pixels)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Mask Area Distribution')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check3_mask_integrity.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 4: Slice-Level Coverage =====
    print("\nCHECK 4: Slice-Level Coverage")
    
    if 'volume_id' in full_df.columns:
        slices_per_volume = full_df.groupby('volume_id').size()
        print(f"  Slices per Volume:")
        print(f"    Mean: {slices_per_volume.mean():.1f}")
        print(f"    Median: {slices_per_volume.median():.1f}")
        print(f"    Range: [{slices_per_volume.min()}, {slices_per_volume.max()}]")
        
        plt.figure(figsize=(10, 5))
        plt.hist(slices_per_volume, bins=30, edgecolor='black', alpha=0.7)
        plt.xlabel('Slices per Volume')
        plt.ylabel('Frequency')
        plt.title('Distribution of Slices per Volume')
        plt.grid(alpha=0.3)
        plt.savefig(f"{output_dir}/check4_slices_per_volume.png", dpi=150, bbox_inches='tight')
        plt.close()
    else:
        print("  Volume ID not available, skipping volume-level checks")
    
    # ===== CHECK 5: HU Within Mask (Physics Features) =====
    print("\nCHECK 5: HU Within Mask (Physics Features)")
    
    print(f"  μ_avg (HU mean in mask):")
    print(f"    COVID:  {full_df[full_df['label']==1]['mu_avg'].mean():.2f} ± {full_df[full_df['label']==1]['mu_avg'].std():.2f}")
    print(f"    Normal: {full_df[full_df['label']==0]['mu_avg'].mean():.2f} ± {full_df[full_df['label']==0]['mu_avg'].std():.2f}")
    
    print(f"  HU_std:")
    print(f"    COVID:  {full_df[full_df['label']==1]['hu_std'].mean():.2f} ± {full_df[full_df['label']==1]['hu_std'].std():.2f}")
    print(f"    Normal: {full_df[full_df['label']==0]['hu_std'].mean():.2f} ± {full_df[full_df['label']==0]['hu_std'].std():.2f}")
    
    # Plot distributions
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    features = ['mu_avg', 'hu_std', 'hu_p50', 'grad_mean', 'mask_fraction', 'hu_p90']
    for idx, feature in enumerate(features):
        ax = axes[idx // 3, idx % 3]
        
        covid_data = full_df[full_df['label']==1][feature].dropna()
        normal_data = full_df[full_df['label']==0][feature].dropna()
        
        ax.violinplot([normal_data, covid_data], positions=[0, 1], showmeans=True)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Normal', 'COVID'])
        ax.set_ylabel(feature)
        ax.set_title(f'{feature} Distribution')
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check5_physics_features.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 6: Outlier Detection =====
    print("\nCHECK 6: Outlier Detection (IQR Method)")
    
    outlier_results = {}
    
    for feature in ['mu_avg', 'hu_std', 'mask_area_pixels', 'grad_mean']:
        data = full_df[feature].dropna()
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = ((data < lower_bound) | (data > upper_bound)).sum()
        outlier_results[feature] = {
            'count': outliers,
            'percentage': outliers / len(data) * 100
        }
        
        print(f"  {feature}: {outliers} outliers ({outliers/len(data)*100:.2f}%)")
    
    # Scatter plot: mu_avg vs mask_area
    plt.figure(figsize=(10, 6))
    covid_mask = full_df['label'] == 1
    
    plt.scatter(full_df[~covid_mask]['mask_area_pixels'], 
                full_df[~covid_mask]['mu_avg'],
                alpha=0.5, s=20, label='Normal', c='blue')
    plt.scatter(full_df[covid_mask]['mask_area_pixels'], 
                full_df[covid_mask]['mu_avg'],
                alpha=0.5, s=20, label='COVID', c='red')
    
    plt.xlabel('Mask Area (pixels)')
    plt.ylabel('μ_avg (Mean HU)')
    plt.title('Mask Area vs Mean HU (Outlier Detection)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(f"{output_dir}/check6_outliers.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 7: Image Quality (Gradient-based) =====
    print("\nCHECK 7: Image Quality (Gradient Magnitude)")
    
    grad_threshold = full_df['grad_mean'].quantile(0.05)  # Bottom 5%
    low_quality = full_df[full_df['grad_mean'] < grad_threshold]
    
    print(f"  Mean gradient magnitude: {full_df['grad_mean'].mean():.2f}")
    print(f"  Low quality threshold (5th percentile): {grad_threshold:.2f}")
    print(f"  Low quality slices: {len(low_quality)} ({len(low_quality)/len(full_df)*100:.2f}%)")
    
    plt.figure(figsize=(10, 5))
    plt.hist([full_df[full_df['label']==0]['grad_mean'].dropna(),
              full_df[full_df['label']==1]['grad_mean'].dropna()],
             bins=50, label=['Normal', 'COVID'], alpha=0.6)
    plt.axvline(grad_threshold, color='red', linestyle='--', label=f'Low Quality (<{grad_threshold:.1f})')
    plt.xlabel('Mean Gradient Magnitude')
    plt.ylabel('Frequency')
    plt.title('Image Quality Distribution')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(f"{output_dir}/check7_image_quality.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 8: Label Distribution Across Splits =====
    print("\n" + "="*70)
    print("CHECK 8: Label Distribution & Split Balance")
    print("="*70)
    
    split_stats = []
    for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        covid = (df['label'] == 1).sum()
        normal = (df['label'] == 0).sum()
        split_stats.append({
            'Split': name,
            'COVID': covid,
            'Normal': normal,
            'Total': len(df),
            'COVID %': covid / len(df) * 100
        })
    
    split_df = pd.DataFrame(split_stats)
    print(split_df.to_string(index=False))
    
    # Chi-square test for independence
    contingency = np.array([[row['COVID'], row['Normal']] for _, row in split_df.iterrows()])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    print(f"\n  Chi-square test (label distribution across splits):")
    print(f"    χ² = {chi2:.4f}, p-value = {p_value:.4f}")
    if p_value > 0.05:
        print(f" No significant difference in label distribution across splits")
    else:
        print(f" WARNING: Significant difference detected!")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.arange(len(split_df))
    width = 0.35
    
    axes[0].bar(x - width/2, split_df['COVID'], width, label='COVID', alpha=0.8)
    axes[0].bar(x + width/2, split_df['Normal'], width, label='Normal', alpha=0.8)
    axes[0].set_xlabel('Split')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Label Distribution Across Splits')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(split_df['Split'])
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    axes[1].bar(split_df['Split'], split_df['COVID %'], alpha=0.8, color='coral')
    axes[1].axhline(50, color='red', linestyle='--', label='Perfect Balance (50%)')
    axes[1].set_xlabel('Split')
    axes[1].set_ylabel('COVID %')
    axes[1].set_title('COVID Percentage by Split')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check8_label_distribution.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    print("\nCHECK 9: Physics Feature Statistical Comparison")
    
    for feature in ['mu_avg', 'hu_std', 'mask_area_pixels', 'grad_mean']:
        print(f"\n  {feature}:")
        covid_data = full_df[full_df['label'] == 1][feature].dropna()
        normal_data = full_df[full_df['label'] == 0][feature].dropna()
        
        print(f"    COVID:  μ={covid_data.mean():.2f}, σ={covid_data.std():.2f}, n={len(covid_data)}")
        print(f"    Normal: μ={normal_data.mean():.2f}, σ={normal_data.std():.2f}, n={len(normal_data)}")
        
        # Mann-Whitney U test
        stat, p_val = mannwhitneyu(covid_data, normal_data, alternative='two-sided')
        print(f"    Mann-Whitney U test: U={stat:.2f}, p={p_val:.4f}")
        if p_val < 0.05:
            print(f"    ✓ Significant difference detected!")
        else:
            print(f"    No significant difference")
    
    # Plot per-label distributions
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, feature in enumerate(['mu_avg', 'hu_std', 'grad_mean']):
        ax = axes[idx]
        
        covid_data = full_df[full_df['label'] == 1][feature].dropna()
        normal_data = full_df[full_df['label'] == 0][feature].dropna()
        
        ax.violinplot([normal_data, covid_data], positions=[0, 1], showmeans=True)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Normal', 'COVID'])
        ax.set_ylabel(feature)
        ax.set_title(f'{feature} by Label')
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check9_physics_comparison.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== FINAL SUMMARY =====
    with open(f"{output_dir}/SANITY_CHECK_SUMMARY.txt", 'w') as f:
        f.write(f"SANITY CHECK SUMMARY\n")
        f.write(f"{'='*70}\n\n")
        f.write(f"Total Samples: {len(full_df):,}\n")
        f.write(f"  Train: {len(train_df):,}\n")
        f.write(f"  Val:   {len(val_df):,}\n")
        f.write(f"  Test:  {len(test_df):,}\n\n")
        
        f.write("CHECK 1: File Integrity\n")
        f.write(f"  Missing files: {len(missing_ct) + len(missing_mu) + len(missing_mask)}\n\n")
        
        f.write("CHECK 2: HU Range\n")
        f.write(f"  Mean HU: {stats_df['mean'].mean():.1f} ± {stats_df['mean'].std():.1f}\n")
        f.write(f"  Extreme values: {outlier_count}\n\n")  # ✅ FIX: Use saved count
        
        f.write("CHECK 3: Mask Integrity\n")
        f.write(f"  Small masks (<1000px): {empty_masks} ({empty_masks/len(full_df)*100:.2f}%)\n\n")
        
        f.write("CHECK 6: Outliers (IQR)\n")
        for feature, result in outlier_results.items():
            f.write(f"  {feature}: {result['count']} ({result['percentage']:.2f}%)\n")
        f.write("\n")
        
        f.write("CHECK 7: Image Quality\n")
        f.write(f"  Low quality slices: {len(low_quality)} ({len(low_quality)/len(full_df)*100:.2f}%)\n\n")
        
        f.write("CHECK 8: Split Balance\n")
        f.write(f"  Chi-square p-value: {p_value:.4f}\n")
        f.write(f"  Balance: {'PASS' if p_value > 0.05 else 'FAIL'}\n\n")
        
        f.write("="*70 + "\n")
        f.write("All sanity check plots saved to: " + output_dir + "\n")
    
    print(f"Results saved to: {output_dir}/")
    print(f"Summary report: {output_dir}/SANITY_CHECK_SUMMARY.txt")


In [ ]:
run_all_sanity_checks(train_df, val_df, test_df)

# Data Loaders for Cohort A

In [ ]:
class CTDataset(Dataset):
    def __init__(self,csv_path):
        self.df = pd.read_csv(csv_path)
        print(f"Loaded {len(self)} samples")
        print(f"  COVID:  {len(self.df[self.df['label']==1]):,}")
        print(f"  Normal: {len(self.df[self.df['label']==0]):,}")


    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        ct = np.load(row['ct_path'])
        mu = np.load(row['mu_path'])
        mask = np.load(row['mask_path'])

        return {
            'ct': torch.FloatTensor(ct).unsqueeze(0),
            'mu': torch.FloatTensor(mu).unsqueeze(0),
            'mask': torch.FloatTensor(mask).unsqueeze(0),
            'label': torch.tensor(row['label'], dtype=torch.long)
        }
        

In [ ]:
def visualize_samples_from_dataset(df, num_samples=6, output_dir='/kaggle/working/visualizations'):
    os.makedirs(output_dir, exist_ok=True)
    
    # Sample equal number from each class
    covid_samples = df[df['label'] == 1].sample(n=num_samples//2, random_state=42)
    normal_samples = df[df['label'] == 0].sample(n=num_samples//2, random_state=42)
    samples = pd.concat([covid_samples, normal_samples]).sample(frac=1, random_state=42)
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    for idx, (_, row) in enumerate(samples.iterrows()):
        # Load data
        ct = np.load(row['ct_path'])
        mu = np.load(row['mu_path'])
        mask = np.load(row['mask_path'])
        
        # Denormalize CT back to HU
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        
        # Apply mask
        masked_ct = ct_hu * mask
        
        label_text = "COVID" if row['label'] == 1 else "NORMAL"
        label_color = 'red' if row['label'] == 1 else 'green'
        
        # Plot 1: Original CT (HU)
        im1 = axes[idx, 0].imshow(ct_hu, cmap='gray', vmin=-1000, vmax=400)
        axes[idx, 0].set_title(f'{label_text}: Original CT (HU)', color=label_color, fontweight='bold')
        axes[idx, 0].axis('off')
        plt.colorbar(im1, ax=axes[idx, 0], fraction=0.046, pad=0.04)
        
        # Plot 2: Lung Mask
        im2 = axes[idx, 1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
        axes[idx, 1].set_title('Lung Mask', fontweight='bold')
        axes[idx, 1].axis('off')
        plt.colorbar(im2, ax=axes[idx, 1], fraction=0.046, pad=0.04)
        
        # Plot 3: Masked/Segmented Lung
        im3 = axes[idx, 2].imshow(masked_ct, cmap='gray', vmin=-1000, vmax=400)
        axes[idx, 2].set_title('Segmented Lung Region', fontweight='bold')
        axes[idx, 2].axis('off')
        plt.colorbar(im3, ax=axes[idx, 2], fraction=0.046, pad=0.04)
        
        # Plot 4: μ (Attenuation) Map
        im4 = axes[idx, 3].imshow(mu * mask, cmap='viridis', vmin=0, vmax=0.3)
        axes[idx, 3].set_title('μ (Attenuation) Map', fontweight='bold')
        axes[idx, 3].axis('off')
        plt.colorbar(im4, ax=axes[idx, 3], fraction=0.046, pad=0.04)
        
        # Add text info
        mask_coverage = mask.sum() / mask.size * 100
        axes[idx, 0].text(10, 210, f"ID: {row['id']}\nCoverage: {mask_coverage:.1f}%", 
                         color='yellow', fontsize=8, bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/sample_visualizations.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSaved visualization to: {output_dir}/sample_visualizations.png")


def visualize_physics_comparison(df, output_dir='/kaggle/working/visualizations'):
    """
    Visualize side-by-side comparison of COVID vs Normal samples
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Get one sample from each class
    covid_sample = df[df['label'] == 1].sample(n=1, random_state=42).iloc[0]
    normal_sample = df[df['label'] == 0].sample(n=1, random_state=42).iloc[0]
    
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    
    for row_idx, (sample, label_text, label_color) in enumerate([
        (normal_sample, "NORMAL", 'green'),
        (covid_sample, "COVID", 'red')
    ]):
        # Load data
        ct = np.load(sample['ct_path'])
        mu = np.load(sample['mu_path'])
        mask = np.load(sample['mask_path'])
        
        # Denormalize CT
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        masked_ct = ct_hu * mask
        
        # Plot 1: Original CT
        im1 = axes[row_idx, 0].imshow(ct_hu, cmap='gray', vmin=-1000, vmax=400)
        axes[row_idx, 0].set_title(f'{label_text}: Original CT', color=label_color, fontweight='bold', fontsize=12)
        axes[row_idx, 0].axis('off')
        plt.colorbar(im1, ax=axes[row_idx, 0], fraction=0.046, pad=0.04)
        
        # Plot 2: Mask
        im2 = axes[row_idx, 1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
        axes[row_idx, 1].set_title('Lung Mask', fontweight='bold', fontsize=12)
        axes[row_idx, 1].axis('off')
        plt.colorbar(im2, ax=axes[row_idx, 1], fraction=0.046, pad=0.04)
        
        # Plot 3: Segmented
        im3 = axes[row_idx, 2].imshow(masked_ct, cmap='gray', vmin=-1000, vmax=400)
        axes[row_idx, 2].set_title('Segmented Lung', fontweight='bold', fontsize=12)
        axes[row_idx, 2].axis('off')
        plt.colorbar(im3, ax=axes[row_idx, 2], fraction=0.046, pad=0.04)
        
        # Plot 4: μ Map
        im4 = axes[row_idx, 3].imshow(mu * mask, cmap='viridis', vmin=0, vmax=0.3)
        axes[row_idx, 3].set_title('μ Attenuation Map', fontweight='bold', fontsize=12)
        axes[row_idx, 3].axis('off')
        plt.colorbar(im4, ax=axes[row_idx, 3], fraction=0.046, pad=0.04)
        
        # Add stats
        lung_pixels = mask > 0.5
        if lung_pixels.sum() > 0:
            hu_mean = ct_hu[lung_pixels].mean()
            hu_std = ct_hu[lung_pixels].std()
            mu_mean = mu[lung_pixels].mean()
            
            stats_text = f"HU: {hu_mean:.1f}±{hu_std:.1f}\nμ: {mu_mean:.4f}\nArea: {mask.sum():.0f}px"
            axes[row_idx, 0].text(10, 210, stats_text, color='yellow', fontsize=9,
                                bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
    
    plt.suptitle('COVID vs NORMAL Comparison', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/covid_vs_normal_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSaved comparison to: {output_dir}/covid_vs_normal_comparison.png")


def visualize_histogram_comparison(df, output_dir='/kaggle/working/visualizations'):
    """
    Visualize HU histograms for COVID vs Normal within lung masks
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Sample 100 from each class
    covid_samples = df[df['label'] == 1].sample(n=min(100, len(df[df['label']==1])), random_state=42)
    normal_samples = df[df['label'] == 0].sample(n=min(100, len(df[df['label']==0])), random_state=42)
    
    covid_hu_values = []
    normal_hu_values = []
    
    print("Computing HU histograms...")
    
    for _, row in tqdm(covid_samples.iterrows(), total=len(covid_samples), desc="COVID samples"):
        ct = np.load(row['ct_path'])
        mask = np.load(row['mask_path'])
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        lung_pixels = mask > 0.5
        covid_hu_values.extend(ct_hu[lung_pixels].flatten())
    
    for _, row in tqdm(normal_samples.iterrows(), total=len(normal_samples), desc="Normal samples"):
        ct = np.load(row['ct_path'])
        mask = np.load(row['mask_path'])
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        lung_pixels = mask > 0.5
        normal_hu_values.extend(ct_hu[lung_pixels].flatten())
    
    # Plot histograms
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Histogram 1: Overlapping
    axes[0].hist(normal_hu_values, bins=100, alpha=0.6, label='Normal', color='green', density=True)
    axes[0].hist(covid_hu_values, bins=100, alpha=0.6, label='COVID', color='red', density=True)
    axes[0].set_xlabel('HU Value', fontsize=12)
    axes[0].set_ylabel('Density', fontsize=12)
    axes[0].set_title('HU Distribution in Lung Regions', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(alpha=0.3)
    axes[0].set_xlim(-1000, 100)
    
    # Add reference lines
    axes[0].axvline(-900, color='blue', linestyle='--', alpha=0.5, label='Lung threshold')
    axes[0].axvline(-500, color='orange', linestyle='--', alpha=0.5, label='GGO range')
    
    # Histogram 2: Side by side boxplot
    axes[1].boxplot([normal_hu_values, covid_hu_values], labels=['Normal', 'COVID'])
    axes[1].set_ylabel('HU Value', fontsize=12)
    axes[1].set_title('HU Value Distribution (Boxplot)', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    # Add statistics
    covid_mean = np.mean(covid_hu_values)
    normal_mean = np.mean(normal_hu_values)
    axes[1].text(1.5, -800, f"COVID μ: {covid_mean:.1f}\nNormal μ: {normal_mean:.1f}", 
                fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/hu_histogram_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSaved histogram to: {output_dir}/hu_histogram_comparison.png")

In [ ]:
visualize_samples_from_dataset(full_balanced_df,num_samples = 10)
visualize_physics_comparison(full_balanced_df)
visualize_histogram_comparison(full_balanced_df)